# 05 — Activation Patching

Logit lens/DLA are correlational: a head can *look* important without
actually *causing* the behavior. Activation patching answers the causal
question directly, and is the closest thing in mech interp to a controlled
experiment — this is the technique to reach for whenever you want to
*evaluate* which part of a model is responsible for a behavior.

**Setup**: two prompts that differ in one key fact, e.g.

- clean: `"John and Mary went to the store, John gave a drink to"` → `" Mary"`
- corrupted: `"John and Mary went to the store, Mary gave a drink to"` → `" John"`

Run the corrupted prompt, then for one specific activation (one head, one
layer, one position), overwrite it with the value from the *clean* run, and
re-run the rest of the forward pass. If the model's output snaps back toward
the clean answer, that activation was causally responsible.


## Shorthand in this notebook

- **IOI** — **I**ndirect **O**bject **I**dentification, the standard name
  in the literature for exactly this task (predicting who receives the
  object in a sentence like "X gave a drink to ___"). It's the most-studied
  example circuit in mech interp because it's simple enough to fully reverse
  engineer but rich enough to involve several head types. This notebook's
  clean/corrupted pair is a minimal IOI example.
- **clean / corrupted** — standard patching terminology. "Clean" is the
  prompt whose behavior you're trying to explain; "corrupted" is a minimally
  different prompt that changes the answer. Patching asks "which activations,
  when moved from clean into the corrupted run, bring the corrupted output
  back to the clean answer?"
- **`logit_diff`** — not a TransformerLens concept, just this notebook's own
  metric: `logit(" Mary") - logit(" John")`. Positive = model favors
  "Mary," negative = favors "John." Using a *difference* rather than a raw
  logit cancels out a lot of unrelated per-prompt noise, which is why it's
  the standard scoring choice for this kind of experiment.
- **`hook_fn`, `fwd_hooks`, `run_with_hooks`** — a "hook" (see notebook 00)
  is a tap point; here we go further and *write* to one instead of just
  reading it. `hook_fn` is the function that does the overwriting,
  `fwd_hooks` is the list of `(hook_name, hook_fn)` pairs to install for one
  forward pass, and `run_with_hooks` is the method that runs the model with
  those hooks active.
- **`partial`** — `functools.partial`, plain Python/standard library, not
  mech-interp-specific. Pre-fills some of a function's arguments (`position`,
  `clean_cache`) so what's left matches the signature TransformerLens expects
  a hook function to have.
- **ACDC** — **A**utomatic **C**ircuit **D**iscovery, an algorithm that
  automates exactly the kind of patching sweep this notebook does by hand,
  to find a full minimal circuit instead of one component at a time.
- **SAE** — **S**parse **A**uto**e**ncoder. A small model trained to
  re-express the residual stream (or any activation) as a sparse combination
  of many more "feature" directions than `d_model` has dimensions — the
  current best tool for turning raw activation directions into
  human-interpretable concepts.
- **RLHF** — **R**einforcement **L**earning from **H**uman **F**eedback,
  the fine-tuning process (using PPO, which you already know) that turns a
  base language model into an instruction-following/chat model.


In [1]:
import torch
from transformer_lens import HookedTransformer, utils
import plotly.express as px
from functools import partial

torch.set_grad_enabled(False)
device = "cuda" if torch.cuda.is_available() else "cpu"
model = HookedTransformer.from_pretrained("gpt2", device=device)


/home/keqingsimp/.conda/envs/mech-interp/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/tmp/ipykernel_68501/2201576842.py:2: DeprecationWarning: The 'utils' module has been deprecated. Please use 'transformer_lens.utilities' instead. Importing from utils.py will be removed in TransformerLens 4.0.
  from transformer_lens import HookedTransformer, utils


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 17281.65it/s]

Loaded pretrained model gpt2 into HookedTransformer


In [2]:
clean_text = "John and Mary went to the store, John gave a drink to"
corrupted_text = "John and Mary went to the store, Mary gave a drink to"

clean_tokens = model.to_tokens(clean_text)
corrupted_tokens = model.to_tokens(corrupted_text)
assert clean_tokens.shape == corrupted_tokens.shape

mary_token = model.to_single_token(" Mary")
john_token = model.to_single_token(" John")

clean_logits, clean_cache = model.run_with_cache(clean_tokens)
corrupted_logits, corrupted_cache = model.run_with_cache(corrupted_tokens)

def logit_diff(logits):
    final = logits[0, -1]
    return (final[mary_token] - final[john_token]).item()

print("clean logit diff (Mary - John):", logit_diff(clean_logits))
print("corrupted logit diff (Mary - John):", logit_diff(corrupted_logits))


clean logit diff (Mary - John): 2.2588157653808594
corrupted logit diff (Mary - John): -3.7372121810913086


## Patch in `resid_pre` at every (layer, position) and measure recovery

For each layer and position, run the corrupted prompt but overwrite
`resid_pre` at that (layer, position) with the clean-run value. Score how
much of the clean/corrupted logit-diff gap gets recovered.


In [3]:
def patch_resid_pre(resid_pre, hook, position, clean_cache):
    resid_pre[:, position, :] = clean_cache[hook.name][:, position, :]
    return resid_pre

n_layers = model.cfg.n_layers
n_pos = clean_tokens.shape[1]
str_tokens = model.to_str_tokens(clean_text)

results = torch.zeros(n_layers, n_pos)
clean_diff = logit_diff(clean_logits)
corrupted_diff = logit_diff(corrupted_logits)

for layer in range(n_layers):
    for position in range(n_pos):
        hook_fn = partial(patch_resid_pre, position=position, clean_cache=clean_cache)
        patched_logits = model.run_with_hooks(
            corrupted_tokens,
            fwd_hooks=[(utils.get_act_name("resid_pre", layer), hook_fn)],
        )
        patched_diff = logit_diff(patched_logits)
        # normalize: 0 = no recovery (still corrupted), 1 = full recovery (matches clean)
        results[layer, position] = (patched_diff - corrupted_diff) / (clean_diff - corrupted_diff)

px.imshow(results, x=str_tokens, y=list(range(n_layers)),
          labels=dict(x="position", y="layer", color="recovery"),
          title="Activation patching: resid_pre recovery of clean logit diff",
          color_continuous_scale="RdBu", color_continuous_midpoint=0).show()


## Reading the result

Bright cells = patching that single (layer, position) activation mostly
restores the clean prediction, i.e. that's *where and when* the model
resolves "who gets the drink." You should see the signal concentrate around
the `" John"`/`" Mary"` name positions in the middle-to-late layers — that's
the causal trace of the model doing coreference resolution.

You can repeat this same recipe (swap `resid_pre` for `hook_z` per head, or
`mlp_out`) to localize the effect to individual attention heads instead of
whole-layer residual stream — that's the natural next step once this pattern
feels familiar.


## Where to go from here

- **ACDC / automated circuit discovery** — automate the patching sweep above
  to find full circuits instead of single components.
- **Sparse autoencoders (SAEs)** — decompose the residual stream into
  human-interpretable *features* instead of reading raw dimensions. Look at
  `sae_lens` (pairs with `transformer_lens`) once you're comfortable here.
- **RLHF/PPO angle** (given your PPO background): compare a base model vs.
  its RLHF-tuned counterpart on the same activation-patching setup — do the
  same circuits still do the same job, or did fine-tuning rewire them? This
  is an open, active research question.
- **Neel Nanda's "200 Concrete Open Problems in Mechanistic Interpretability"**
  and the **ARENA mech interp curriculum** are the standard next stops —
  worth reading once these four notebooks feel natural rather than before.
